In [4]:
library(Seurat)
library(sceasy)
convert_anndata_to_seurat <- function(h5ad_path) {
  # Check if the necessary packages are installed
  if (!requireNamespace("Seurat", quietly = TRUE)) {
    stop("Seurat package is not installed. Please install it using install.packages('Seurat')")
  }
  if (!requireNamespace("sceasy", quietly = TRUE)) {
    stop("sceasy package is not installed. Please install it using install.packages('sceasy')")
  }

  # Convert the AnnData object to a Seurat object
  sceasy::convertFormat(
    h5ad_path,
    from = "anndata",
    to = "seurat",
    outFile = "temp.rds"
  )

  # Read the Seurat object from the temporary file
  seurat_object <- readRDS("temp.rds")


  # Remove the temporary file
  unlink("temp.rds")

  return(seurat_object)
}

In [5]:
perform_dea_hardcoded <- function(seurat_object,
                                  condition_column = "Condition",
                                  ident.1 = NULL,
                                  ident.2 = NULL, 
                                output_file = NULL) {
  # Check if the Seurat object is valid
  if (!inherits(seurat_object, "Seurat")) {
    stop("Input must be a Seurat object.")
  }

  # Check if the condition column exists in the metadata
  if (!(condition_column %in% colnames(seurat_object[[]]))) {
    stop(
      "Condition column '",
      condition_column,
      "' not found in Seurat object metadata."
    )
  }

  # Set the identity class
  Idents(seurat_object) <- "condition"

  # Perform differential expression analysis using FindMarkers with hardcoded parameters
  dea_results <- FindMarkers(
    seurat_object,
    ident.1 = ident.1,
    ident.2 = ident.2,
    group.by = condition_column,
    min.pct = 0.25,
    logfc.threshold = 0.3,
    test.use = "wilcox",
    only.pos = FALSE,
    pseudocount.use = 1,
  )
# Save the results to a CSV file if an output file is specified
  if (!is.null(output_file)) {
    write.csv(dea_results, file = output_file)
  }
  return(dea_results)
}

In [6]:
filter_dea_results <- function(dea_results,
                               p_adj_threshold = 0.05,
                               output_file = NULL) {
  # Check if the input is a data frame
  if (!is.data.frame(dea_results)) {
    stop("Input 'dea_results' must be a data frame.")
  }

  # Check if the p_adj column exists
  if (!("p_val_adj" %in% colnames(dea_results))) {
    stop("Column 'p_adj' not found in 'dea_results'.")
  }

  # Filter the results based on the adjusted p-value threshold
  filtered_results <-
    dea_results[dea_results$p_val_adj <= p_adj_threshold, , drop = FALSE]

  # Save the results to a CSV file if an output file is specified
  if (!is.null(output_file)) {
    write.csv(filtered_results, file = output_file)
  }

  return(filtered_results)
}

find_gene_in_dea <- function(dea_results, gene_name, ignore_case = TRUE) {
  # Check if the input is a data frame
  if (!is.data.frame(dea_results)) {
    stop("Input 'dea_results' must be a data frame.")
  }

  # Check if the data frame has row names
  if (is.null(rownames(dea_results))) {
    stop("The 'dea_results' data frame must have gene names as row names.")
  }

  # Find the gene in the row names
  if (ignore_case) {
    gene_index <-
      which(stringr::str_to_lower(rownames(dea_results)) == stringr::str_to_lower(gene_name))
  } else {
    gene_index <- which(rownames(dea_results) == gene_name)
  }

  # Return the gene information or NULL if not found
  if (length(gene_index) > 0) {
    return(dea_results[gene_index, , drop = FALSE]) # Return as a data frame
  } else {
    return(NULL)
  }
}

In [ ]:
list.files("../Dataset/MatriciAddestramento")

In [30]:
#CSF BCELLS
bcells = convert_anndata_to_seurat("../Dataset/MatriciAddestramento/CSF_BCELLS.h5ad")
dea = perform_dea_hardcoded(bcells, "condition", "MS", "CTRL")
bcells_dea = filter_dea_results(dea, p_adj_threshold = 0.05, "./Store/DEA/CSF_BCells.csv") 

X -> counts

Warning message in mean.fxn(object[features, cells.1, drop = FALSE]):
“NaNs produced”
Warning message in mean.fxn(object[features, cells.2, drop = FALSE]):
“NaNs produced”


In [10]:
#CSF CD4
cd4 = convert_anndata_to_seurat("../Dataset/MatriciAddestramento/CD4_CSF_SAMU.h5ad")
dea = perform_dea_hardcoded(cd4, "condition", "MS", "CTRL")
cd4_dea = filter_dea_results(dea, p_adj_threshold = 0.05, "Store/DEA/CSF_CD4.csv") 

X -> counts

Warning message:
“The `slot` argument of `GetAssayData()` is deprecated as of SeuratObject 5.0.0.
ℹ Please use the `layer` argument instead.
ℹ The deprecated feature was likely used in the Seurat package.
  Please report the issue at <https://github.com/satijalab/seurat/issues>.”
Warning message in mean.fxn(object[features, cells.1, drop = FALSE]):
“NaNs produced”
Warning message in mean.fxn(object[features, cells.2, drop = FALSE]):
“NaNs produced”
Warning message:
“`PackageCheck()` was deprecated in SeuratObject 5.0.0.
ℹ Please use `rlang::check_installed()` instead.
ℹ The deprecated feature was likely used in the Seurat package.
  Please report the issue at <https://github.com/satijalab/seurat/issues>.”
For a (much!) faster implementation of the Wilcoxon Rank Sum Test,
(default method for FindMarkers) please install the presto package
--------------------------------------------
install.packages('devtools')
devtools::install_github('immunogenomics/presto')
--------------

In [ ]:
#PBMC CD4
cd4 = convert_anndata_to_seurat("../Dataset/MatriciAddestramento/PBMC_CD4TNONAIVE.h5ad")
dea = perform_dea_hardcoded(cd4, "condition", "MS", "CTRL")
cd4_dea = filter_dea_results(dea, p_adj_threshold = 0.05, "Store/DEA/PBMC_CD4.csv") 

X -> counts



In [31]:
#PBMC BCELLS
bcells = convert_anndata_to_seurat("../Dataset/MatriciAddestramento/PBMC_BCELLS_2DATASET.h5ad")
dea = perform_dea_hardcoded(bcells, "condition", "MS", "CTRL")
bcells_dea = filter_dea_results(dea, p_adj_threshold = 0.05, "Store/DEA/PBMC_BCELLS.csv") 

X -> counts

Warning message in mean.fxn(object[features, cells.1, drop = FALSE]):
“NaNs produced”
Warning message in mean.fxn(object[features, cells.2, drop = FALSE]):
“NaNs produced”


In [9]:
py_require("anndata")    # Declare jax is a requirement